# 03 · Retrieval fidelity (Experiment 1, Table 1)

Does a query encoder the board can run select grains as the teacher (the music2latent encoder) would? Experiment 1
compares five query encoders. Three can run on the board: a **ridge map** from 80 log-mel bands and 9 spectral
descriptors, the **student** (MLP) and a 128-unit **GRU**. Two are references: the **teacher with a second random
seed**, and query **MFCCs** matched against corpus MFCCs (a lower bound). The corpus side is always the pack's teacher
latents, and only the query side changes (`distillation/evaluate.py`).

Metrics, all against the teacher's similarity ordering of every grain in the corpus:

* **Spearman ρ** between the encoder's and the teacher's similarity orderings of all grains, per frame, averaged.
* **Percentile rank (Pct.)** of the grain the encoder selects at the instrument's τ = 0.05, in the teacher's ordering
  (.95: more similar than 95 % of the corpus).
* **Selection agreement (Agr.)**: the share of frames on which the encoder selects the same grain as the teacher,
  with the same seed (matched random draws: `Generator.choice` consumes one uniform per frame).
* **Hub share (Hub)**: the share of selections taken by the five most-selected grains (`corpus_builder/match.py`).

Part 1 computes them on the demo pack. Part 2 reproduces Table 1 from the per-query metrics of the paper's run
(`results/exp1/`, regenerated on 23 Sep 2026 from the committed weights, see `results/README.md`).

In [1]:
import json

import numpy as np
import torch

from analysis import demo, exp1, report
from corpus_builder import elkgrain
from corpus_builder.match import Matcher
from distillation import ab_common, features as F, ridgefit as R, student as S
from distillation.evaluate import sims_matrix, spearman_rows

pk = elkgrain.ElkGrainPack(demo.ensure_pack())
book, centre = np.asarray(pk.emb), np.asarray(pk.centre)
tab = ab_common.pack_pair_table(pk, demo.CORPUS_DIR, F.frame_features)
mf = tab["mfcc13"]
mcentre = mf.mean(0)
mbook = (mf - mcentre) / (np.linalg.norm(mf - mcentre, axis=1, keepdims=True) + 1e-8)   # MFCC table, same matching math
queries = demo.encode_queries()
print(pk, "|", {q: v["lat"].shape[1] for q, v in queries.items()}, "frames per query")

ElkGrainPack(demo.elkgrain: 633 grains, unit=1, hop48=4486, margin48=480, 3 files, 59.3s audio, 6.0 MB) | {'qp1': 85, 'qt1': 85, 'qx1': 85} frames per query


## Part 1 · The metrics on the demo pack

The query encoders are the paper's: the student and GRU from `weights/`, the pooled ridge map the board ships
(`weights/ridge_pooled.json`), MFCCs, and the teacher.

In [2]:
mlp, mu, sd = S.mlp_from_json()
gru, gmu, gsd = S.gru_from_json()
rj = json.loads((S.REPO_WEIGHTS / "ridge_pooled.json").read_text())
ridge = {k: np.asarray(rj[k], np.float32) for k in ("mu", "sd", "W", "b")}

def encode(q):
    with torch.no_grad():
        b = mlp(torch.tensor((q["mel80"] - mu) / sd)).numpy().T
        g = gru(torch.tensor((q["mel80"] - gmu) / gsd)[None])[0][0].numpy().T
    return {"O": q["lat"], "F": q["mfcc13"].T, "Apool": R.apply_ridge(ridge, q["feat89"]).T.astype(np.float32),
            "Bgru": g.astype(np.float32), "B": b.astype(np.float32)}

def percentiles(SO):
    order = SO.argsort(1)
    pct = np.empty_like(SO)
    ar = (np.arange(SO.shape[1], dtype=np.float32) + 1) / SO.shape[1]
    for t in range(len(SO)):
        pct[t, order[t]] = ar
    return pct

TAU, m_lat, m_mfcc = 0.05, Matcher(book, centre, 1), Matcher(mbook.astype(np.float32), mcentre.astype(np.float32), 1)
acc = {}
for qid, q in queries.items():
    enc = encode(q)
    SO = sims_matrix(book, centre, q["lat"]); pct = percentiles(SO); t = np.arange(len(SO))
    picks_O = np.asarray(m_lat.select(q["lat"], TAU, rng=np.random.default_rng(0)).picks)
    for e, target in list(enc.items()) + [("O2", q["lat"])]:
        mm, bk, ce = (m_mfcc, mbook, mcentre) if e == "F" else (m_lat, book, centre)
        r = mm.select(target, TAU, rng=np.random.default_rng(1 if e == "O2" else 0))
        p = np.asarray(r.picks)
        row = {"rho": None if e in ("O", "O2") else float(np.mean(spearman_rows(sims_matrix(bk, ce, target), SO))),
               "pct": float(np.mean(pct[t, p])), "agr": float(np.mean(p == picks_O)), "hub": r.hubness}
        for k, v in row.items():
            acc.setdefault(e, {}).setdefault(k, []).append(v)
labels = {"O": "Teacher (reference seed)", "O2": "Teacher, second seed", "F": "MFCC cosine, no map",
          "Apool": "Ridge map", "Bgru": "GRU-128", "B": "Student (MLP)"}
rows = [{"enc": labels[e], **{k: (None if v[0] is None else float(np.mean(v))) for k, v in acc[e].items()}}
        for e in ("O", "O2", "F", "Apool", "Bgru", "B")]
report.table(rows, [("enc", "Query encoder (demo pack, 3 queries)"), ("rho", "ρ", ".3f"), ("pct", "Pct.", ".3f"),
                    ("agr", "Agr.", ".3f"), ("hub", "Hub", ".2f")])

| Query encoder (demo pack, 3 queries) | ρ | Pct. | Agr. | Hub |
|---|---|---|---|---|
| Teacher (reference seed) | — | 0.988 | 1.000 | 0.16 |
| Teacher, second seed | — | 0.989 | 0.176 | 0.15 |
| MFCC cosine, no map | 0.578 | 0.877 | 0.051 | 0.14 |
| Ridge map | 0.709 | 0.906 | 0.098 | 0.21 |
| GRU-128 | 0.773 | 0.938 | 0.098 | 0.20 |
| Student (MLP) | 0.783 | 0.936 | 0.086 | 0.21 |

### Why the teacher agrees with itself on only a few per cent of selections

At τ = 0.05 the instrument samples among near-best grains (so that repeated phrases vary): each frame's selection
probability is p ∝ exp(−(1 − cos) / τ). Two independent draws from the same distribution coincide with probability
Σ p², so even the teacher with a second seed agrees with itself on that share of frames only. The effective number of
candidates, exp(entropy of p), shows how many grains share the probability mass. Agreement therefore measures the
sampling, not the encoder, and the paper reports it without ranking by it.

In [3]:
def softmax_rows(SO, tau):
    logits = -(1.0 - SO) / tau
    logits -= logits.max(1, keepdims=True)
    P = np.exp(logits)
    return P / P.sum(1, keepdims=True)

rows = []
for tau in (0.0, 0.03, 0.05, 0.15):
    obs, expct, eff = [], [], []
    for q in queries.values():
        a = m_lat.select(q["lat"], tau, rng=np.random.default_rng(0)).picks
        b = m_lat.select(q["lat"], tau, rng=np.random.default_rng(1)).picks
        obs.append(np.mean(np.asarray(a) == np.asarray(b)))
        if tau > 0:
            P = softmax_rows(sims_matrix(book, centre, q["lat"]), tau)
            expct.append(np.mean((P ** 2).sum(1)))
            eff.append(np.mean(np.exp(-(P * np.log(P + 1e-30)).sum(1))))
    rows.append({"tau": tau, "obs": float(np.mean(obs)), "exp": float(np.mean(expct)) if expct else 1.0,
                 "eff": float(np.mean(eff)) if eff else 1.0})
report.table(rows, [("tau", "τ", "g"), ("obs", "Teacher vs teacher, second seed: agreement", ".3f"),
                    ("exp", "Expected, Σ p²", ".3f"), ("eff", "Effective candidates, exp(H)", ".1f")])

| τ | Teacher vs teacher, second seed: agreement | Expected, Σ p² | Effective candidates, exp(H) |
|---|---|---|---|
| 0 | 1.000 | 1.000 | 1.0 |
| 0.03 | 0.329 | 0.372 | 7.5 |
| 0.05 | 0.176 | 0.183 | 17.7 |
| 0.15 | 0.012 | 0.016 | 133.0 |

On the demo pack (633 grains) a handful of grains carry the mass. The paper's corpora hold 4,554 to 12,018 grains,
so at τ = 0.05 the teacher agrees with itself on 2.6 % of selections (Table 1): agreement is low for every encoder
and not the criterion. ρ and Pct. are.

## Part 2 · Table 1 from the paper's per-query metrics

`results/exp1/metrics_per_query.csv` holds one row per corpus × query × encoder: 4 corpora (percussive, tonal,
textural, field recordings) × 12 held-out 8 s queries × 7 encoders (A, the per-corpus ridge map, is not in Table 1).
ρ and Pct. are means over the three character corpora, Agr. and Hub over all 48 query–corpus pairs
(`analysis/exp1.py`).

In [4]:
rows = exp1.load()
print(len(rows), "rows,", "corpora:", sorted({r["corpus"] for r in rows}), "| queries:", len({r["query"] for r in rows}),
      "| encoders:", sorted({r["encoder"] for r in rows}))
t1 = exp1.table1(rows)
view = []
for r in t1:
    p = exp1.PAPER[r["encoder"]]
    macs = r["macs"]
    view.append({"label": r["label"], "macs": "—" if macs is None else (f"{macs / 1e3:.1f} k" if macs < 1e4 else f"{macs / 1e6:.2f} M"),
                 "rho": exp1.fmt3(r["rho"]), "pct": exp1.fmt3(r["pct"]), "agr": exp1.fmt3(r["agr"]), "hub": exp1.fmt2(r["hub"]),
                 "paper": " / ".join(x or "—" for x in (p["rho"], p["pct"], p["agr"], p["hub"]))})
report.table(view, [("label", "Query encoder"), ("macs", "MACs"), ("rho", "ρ"), ("pct", "Pct."), ("agr", "Agr."),
                    ("hub", "Hub"), ("paper", "Paper: ρ / Pct. / Agr. / Hub")])
teacher = t1[0]
print(f"teacher row: Pct. {teacher['pct']:.5f} with the second seed, {teacher['pct_seed0']:.5f} with the reference seed, "
      f"Hub {teacher['hub']:.4f} / {teacher['hub_seed0']:.4f}")

336 rows, corpora: ['field', 'perc', 'text', 'tonal'] | queries: 12 | encoders: ['A', 'Apool', 'B', 'Bgru', 'F', 'O', 'O2']


| Query encoder | MACs | ρ | Pct. | Agr. | Hub | Paper: ρ / Pct. / Agr. / Hub |
|---|---|---|---|---|---|---|
| Teacher, second seed | — | --- | .982 | .026 | .17 | — / .978 / .026 / .17 |
| MFCC cosine, no map | — | .585 | .841 | .007 | .12 | .585 / .841 / .007 / .12 |
| Ridge map | 5.7 k | .811 | .937 | .039 | .20 | .811 / .937 / .039 / .20 |
| GRU-128 | 0.09 M | .841 | .949 | .045 | .21 | .841 / .949 / — / — |
| Student (MLP) | 0.10 M | .843 | .950 | .047 | .22 | .843 / .950 / .047 / .22 |

teacher row: Pct. 0.98170 with the second seed, 0.97853 with the reference seed, Hub 0.1696 / 0.1681


**One cell differs.** The paper prints .978 for the teacher's percentile rank. The per-query logs give .97853 with the
reference seed (the value the paper intended: the mean of .9753, .9824 and .9779 over the three corpora), which rounds
to **.979**. The printed .978 is the mean of the per-corpus values after rounding them to three decimals. The row is labelled
"second seed", and with the second seed the percentile rank is **.982** (hub share .17 either way). Every other cell
equals the paper at its printed precision.

### Per corpus

In [5]:
corp = ("perc", "tonal", "text", "field")
per = []
for e in ("B", "Bgru", "Apool", "F", "O"):
    row = {"enc": exp1.PAPER.get(e, {"label": "Teacher (reference seed)"})["label"]}
    for c in corp:
        rho = exp1.cell(rows, e, "spearman", (c,))[0]
        row[f"rho_{c}"] = None if e == "O" else rho
        row[f"pct_{c}"] = exp1.cell(rows, e, "live_pctile", (c,))[0]
    per.append(row)
report.table(per, [("enc", "Encoder")] + [(f"rho_{c}", f"ρ {c}", ".3f") for c in corp] + [(f"pct_{c}", f"Pct. {c}", ".3f") for c in corp])
leads = all(exp1.cell(rows, "B", "spearman", (c,))[0] > exp1.cell(rows, "Apool", "spearman", (c,))[0] >
            exp1.cell(rows, "F", "spearman", (c,))[0] for c in corp)
print("student > ridge map > MFCC on every corpus:", leads)
b_pct = [exp1.cell(rows, "B", "live_pctile", (c,))[0] for c in corp]
print(f"student's selections: top {100 * (1 - max(b_pct)):.1f} to {100 * (1 - min(b_pct)):.1f} % of the teacher's ordering")

| Encoder | ρ perc | ρ tonal | ρ text | ρ field | Pct. perc | Pct. tonal | Pct. text | Pct. field |
|---|---|---|---|---|---|---|---|---|
| Student (MLP) | 0.867 | 0.817 | 0.844 | 0.788 | 0.935 | 0.955 | 0.960 | 0.930 |
| GRU-128 | 0.869 | 0.809 | 0.845 | 0.781 | 0.937 | 0.952 | 0.958 | 0.933 |
| Ridge map | 0.840 | 0.777 | 0.816 | 0.756 | 0.924 | 0.940 | 0.948 | 0.919 |
| MFCC cosine, no map | 0.677 | 0.583 | 0.494 | 0.546 | 0.843 | 0.823 | 0.858 | 0.818 |
| Teacher (reference seed) | — | — | — | — | 0.975 | 0.982 | 0.978 | 0.979 |

student > ridge map > MFCC on every corpus: True
student's selections: top 4.0 to 7.0 % of the teacher's ordering


### Agreement at other τ, and where the student errs

In [6]:
sweep = [{"enc": exp1.PAPER.get(e, {"label": e})["label"],
          **{f"t{t}": exp1.cell(rows, e, f"agree@{t:g}")[0] for t in (0, 0.03, 0.05, 0.15)}} for e in ("O2", "B", "Bgru", "Apool", "F")]
report.table(sweep, [("enc", "Encoder (48 pairs)")] + [(f"t{t}", f"Agr. τ = {t:g}", ".3f") for t in (0, 0.03, 0.05, 0.15)])

import csv
ee = list(csv.DictReader(open(demo.REPO / "results/exp1/embedding_error.csv")))
err = {e: {k: np.mean([float(r[k]) for r in ee if r["encoder"] == e]) for k in ("mean", "onset_mean", "sustain_mean")}
       for e in ("B", "Bgru", "Apool")}
report.table([{"enc": exp1.PAPER[e]["label"], **v} for e, v in err.items()],
             [("enc", "Encoder"), ("mean", "1 − centred cosine, all frames", ".3f"),
              ("onset_mean", "onset frames (top-decile flux)", ".3f"), ("sustain_mean", "other frames", ".3f")])

| Encoder (48 pairs) | Agr. τ = 0 | Agr. τ = 0.03 | Agr. τ = 0.05 | Agr. τ = 0.15 |
|---|---|---|---|---|
| Teacher, second seed | 1.000 | 0.128 | 0.026 | 0.000 |
| Student (MLP) | 0.158 | 0.098 | 0.047 | 0.010 |
| GRU-128 | 0.128 | 0.088 | 0.045 | 0.007 |
| Ridge map | 0.143 | 0.085 | 0.039 | 0.009 |
| MFCC cosine, no map | 0.014 | 0.008 | 0.007 | 0.005 |

| Encoder | 1 − centred cosine, all frames | onset frames (top-decile flux) | other frames |
|---|---|---|---|
| Student (MLP) | 0.266 | 0.295 | 0.262 |
| GRU-128 | 0.279 | 0.310 | 0.276 |
| Ridge map | 0.291 | 0.319 | 0.288 |

## Numbers of the paper this notebook reproduces

In [7]:
T = {r["encoder"]: r for r in t1}
items = []
for e in ("O2", "F", "Apool", "Bgru", "B"):
    p, r = exp1.PAPER[e], T[e]
    for key, f in (("rho", exp1.fmt3), ("pct", exp1.fmt3), ("agr", exp1.fmt3), ("hub", exp1.fmt2)):
        if p[key] is None:
            continue
        here = f(r[key])
        if e == "O2" and key in ("pct", "hub"):         # the paper printed the reference seed's values in this row
            here = f"{f(r[key + '_seed0'])} (seed 1: {here})"
            items.append((f"Table 1, {p['label']}, {key}", p[key], here, f(r[key + "_seed0"]) == p[key]))
        else:
            items.append((f"Table 1, {p['label']}, {key}", p[key], here, here == p[key]))
items += [
    ("abstract: ρ student / ridge / MFCC", ".84 / .81 / .58",
     " / ".join(f"{T[e]['rho']:.2f}".lstrip("0") for e in ("B", "Apool", "F")),
     [f"{T[e]['rho']:.2f}".lstrip("0") for e in ("B", "Apool", "F")] == [".84", ".81", ".58"]),
    ("abstract: selections in the top 4–7 % of the teacher's ordering", "4–7 %",
     f"{100 * (1 - max(b_pct)):.1f}–{100 * (1 - min(b_pct)):.1f} %", round(100 * (1 - max(b_pct))) == 4 and round(100 * (1 - min(b_pct))) == 7),
    ("Results: student leads ridge and MFCC on every corpus", "yes", str(leads), leads),
    ("Results: student vs teacher percentile", ".95 against .978", f".{round(1000 * T['B']['pct'])} against {exp1.fmt3(T['O2']['pct_seed0'])}",
     exp1.fmt3(T["O2"]["pct_seed0"]) == ".978"),
    ("Discussion: onset vs other frames, 1 − cosine", ".295 against .262",
     f"{exp1.fmt3(err['B']['onset_mean'])} against {exp1.fmt3(err['B']['sustain_mean'])}",
     (exp1.fmt3(err["B"]["onset_mean"]), exp1.fmt3(err["B"]["sustain_mean"])) == (".295", ".262")),
    ("Section 4: the teacher agrees with itself rarely at τ = 0.05", "low for every encoder",
     f"{exp1.fmt3(T['O2']['agr'])}", T["O2"]["agr"] < 0.05),
]
ok = report.checks(items)
print("all equal:", ok, "| differing:", [w for w, p, h, good in items if not good])

| Paper element | Printed in the paper | Computed here | Equal |
|---|---|---|---|
| Table 1, Teacher, second seed, pct | .978 | .979 (seed 1: .982) | ✗ differs |
| Table 1, Teacher, second seed, agr | .026 | .026 | ✓ |
| Table 1, Teacher, second seed, hub | .17 | .17 (seed 1: .17) | ✓ |
| Table 1, MFCC cosine, no map, rho | .585 | .585 | ✓ |
| Table 1, MFCC cosine, no map, pct | .841 | .841 | ✓ |
| Table 1, MFCC cosine, no map, agr | .007 | .007 | ✓ |
| Table 1, MFCC cosine, no map, hub | .12 | .12 | ✓ |
| Table 1, Ridge map, rho | .811 | .811 | ✓ |
| Table 1, Ridge map, pct | .937 | .937 | ✓ |
| Table 1, Ridge map, agr | .039 | .039 | ✓ |
| Table 1, Ridge map, hub | .20 | .20 | ✓ |
| Table 1, GRU-128, rho | .841 | .841 | ✓ |
| Table 1, GRU-128, pct | .949 | .949 | ✓ |
| Table 1, Student (MLP), rho | .843 | .843 | ✓ |
| Table 1, Student (MLP), pct | .950 | .950 | ✓ |
| Table 1, Student (MLP), agr | .047 | .047 | ✓ |
| Table 1, Student (MLP), hub | .22 | .22 | ✓ |
| abstract: ρ student / ridge / MFCC | .84 / .81 / .58 | .84 / .81 / .58 | ✓ |
| abstract: selections in the top 4–7 % of the teacher's ordering | 4–7 % | 4.0–7.0 % | ✓ |
| Results: student leads ridge and MFCC on every corpus | yes | True | ✓ |
| Results: student vs teacher percentile | .95 against .978 | .950 against .979 | ✗ differs |
| Discussion: onset vs other frames, 1 − cosine | .295 against .262 | .295 against .262 | ✓ |
| Section 4: the teacher agrees with itself rarely at τ = 0.05 | low for every encoder | .026 | ✓ |

all equal: False | differing: ['Table 1, Teacher, second seed, pct', 'Results: student vs teacher percentile']
